Обучим на концах CLIP свой классификатор.

In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from torchvision import datasets
import torchvision
from transformers import CLIPProcessor, CLIPModel

/home/shkaf2m/Desktop/ml-isp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(10),
    torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = Imagenette(root = './data', split = 'train', download = True, transform = train_transforms)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers = 7)

In [3]:
validation_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

validation_dataset = Imagenette(root = './data', split = 'val', download = True, transform = validation_transforms)
validation_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False, num_workers = 7)

In [4]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
class_names = [labels[0] for labels in train_dataset.classes]
print(class_names)

# Заморозка энкодера визуальной составляющей
model.vision_model.requires_grad_(False)

# Хотим сделать классификатор 10 классов => Добавить слой 10 классов
classifier = torch.nn.Linear(512, len(class_names)).to(DEVICE)
model.to(DEVICE)

['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute']


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [6]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

In [ ]:
from tqdm import tqdm
# Тренировка
epoch_count = 5
for epoch in range(1, epoch_count + 1):
  print(f"Epoch: {epoch} / {epoch_count}")
  model.eval()
  classifier.train()
  
  curr_loss = 0
  curr_corrects = 0
  for inputs, labels in tqdm(train_loader, "Training"):
    inputs = inputs.to(DEVICE)
    labels = labels.to(DEVICE)
    
    optimizer.zero_grad()
    
    # Визуальный энкодер заморожен
    with torch.no_grad():
      # Получение эмбендингов картинок
      image_features = model.get_image_features(pixel_values = inputs)
    
    outputs = classifier(image_features)
    _, preds = torch.max(outputs, 1)
    loss = criterion(outputs, labels)
    
    loss.backward()
    optimizer.step()
    
    curr_loss += loss.item() * inputs.size(0)
    curr_corrects += torch.sum(preds == labels.data)
  epoch_loss = curr_loss / len(train_loader.dataset)
  epoch_acc = curr_corrects.double() / len(train_loader.dataset)

  # Валидация
  model.eval()
  classifier.eval()
  val_corrects = 0

  for images, labels in tqdm(validation_loader, "Validation"):
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)
    with torch.no_grad():
      image_features = model.get_image_features(pixel_values=images)
    outputs = classifier(image_features)
    _, preds = torch.max(outputs, 1)
    loss = criterion(outputs, labels)

    val_corrects += torch.sum(preds == labels.data)

  val_acc = val_corrects.double() / len(validation_loader.dataset)
  print(f"Train Loss: {epoch_loss}, Train Acc: {epoch_acc}")
  print(f"Val Acc: {val_acc}")

    

Epoch: 1 / 5


Validation: 100%|██████████| 123/123 [00:27<00:00,  4.45it/s]


Train Loss: 0.627025055663741, Train Acc: 0.8941810117224628
Val Acc: 0.9872611464968152
Epoch: 2 / 5


Validation: 100%|██████████| 123/123 [00:27<00:00,  4.46it/s]


Train Loss: 0.248281651692187, Train Acc: 0.9430774104974126
Val Acc: 0.9887898089171974
Epoch: 3 / 5


...

Train Loss: 0.1955663303139721, Train Acc: 0.9487802302249445
Val Acc: 0.9882802547770699
Epoch: 4 / 5


Validation: 100%|██████████| 123/123 [00:27<00:00,  4.44it/s]


Train Loss: 0.19045906969430593, Train Acc: 0.942126940542824
Val Acc: 0.9905732484076433
Epoch: 5 / 5


Validation: 100%|██████████| 123/123 [00:27<00:00,  4.44it/s]

Train Loss: 0.17089421685093817, Train Acc: 0.9504699545886577
Val Acc: 0.9910828025477706


In [8]:
torch.save(classifier.state_dict(), 'clip_classifier_head.pth')

In [11]:
def ValidateModel(model, device, data_loader):
  model.eval()
  true_labels = torch.tensor([]).to(DEVICE)
  predicted_labels = torch.tensor([]).to(DEVICE)

  with torch.no_grad():
    for images, labels in tqdm(validation_loader, "Validation"):
      images = images.to(DEVICE)
      labels = labels.to(DEVICE)

      image_features = model.get_image_features(pixel_values=images)
      outputs = classifier(image_features)
      _, predicted = torch.max(outputs.data, 1)

      true_labels = torch.cat((true_labels, labels), 0)
      predicted_labels = torch.cat((predicted_labels, predicted), 0)
  return true_labels, predicted_labels

from sklearn.metrics import f1_score

model = model.to(DEVICE)
model.eval()

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = validation_loader)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

Validation:   0%|          | 0/123 [00:00<?, ?it/s]

Validation: 100%|██████████| 123/123 [00:27<00:00,  4.44it/s]

F1 Score:  0.9910247364967661
